# 65 · Post-E50 Baseline Inventory and Task Matrix

**Estado cero de IA después de la entrega E50.**

Este notebook es un **inventario reproducible**, no un documento escrito a mano. Cada tabla y
cada valor mostrado aquí se deriva de la inspección de archivos locales del repositorio
(configuración, contratos de código, model cards, manifests) en el momento de la ejecución.

**Este notebook NO entrena modelos, NO descarga datasets y NO modifica checkpoints.**

Objetivo: responder, de forma auditable —

- qué datasets existen;
- qué modelos existen;
- qué checkpoints existen;
- qué tareas clínicas/de investigación contempla actualmente el sistema;
- qué serie o modalidad necesita cada tarea;
- qué métricas conocidas tiene cada modelo;
- qué estado de madurez tiene cada tarea;
- qué limitaciones están documentadas;
- qué trabajo Post-E50 corresponde realizar sobre cada capacidad.

Rama: `research/post-e50-ai-vnext` · Base congelada E50: `90f4a8e076cc18efa9ebce10c604e1c3d1f3e35c`


In [1]:
# --- Setup: repository root, allowed write scope, git identity ---
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    # Walk upward from the notebook location until we find repo markers.
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ai_service").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook location")


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The notebook may only write inside these three trees (quality gate, section 16).
ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
BASELINE_DIR = REPO_ROOT / "artifacts" / "post_e50" / "baseline"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"

warnings: list[str] = []
limitations: list[str] = []


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS):
        raise RuntimeError(f"Refusing to write outside allowed Post-E50 trees: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_git(*args: str) -> str:
    result = subprocess.run(
        ["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True
    )
    return result.stdout.strip()


GIT_BRANCH = run_git("branch", "--show-current")
GIT_COMMIT = run_git("rev-parse", "HEAD")
GIT_STATUS_PORCELAIN = run_git("status", "--porcelain")
GENERATED_AT = datetime.now(timezone.utc).isoformat()

print("REPO_ROOT:", REPO_ROOT)
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("GENERATED_AT:", GENERATED_AT)
print("Working tree clean:", GIT_STATUS_PORCELAIN == "")


REPO_ROOT: C:\Users\enzoa\OneDrive\Documentos\1.ReposGitHub-Backends\PFI_MVP_Juntos\PFI_MVPTest_Enzo_AImodule\.claude\worktrees\post-e50-baseline-inventory-cce171
GIT_BRANCH: research/post-e50-ai-vnext
GIT_COMMIT: 90f4a8e076cc18efa9ebce10c604e1c3d1f3e35c
GENERATED_AT: 2026-08-17T22:33:56.278490+00:00
Working tree clean: False


## 1. Fuentes locales inspeccionadas

Se listan a continuación los directorios y archivos que este notebook recorre para construir
el inventario. No se navega Internet en ningún momento de esta fase.


In [2]:
SOURCE_DIRS = [
    "config",
    "models",
    "models/final",
    "models/subarticular",
    "docs",
    "docs/contracts",
    "backlogProducto",
    "notebooks",
    "ai_service/pfi_ai_service/contracts",
    "ai_service/pfi_ai_service",
    "src",
]

source_inventory = []
for rel in SOURCE_DIRS:
    p = REPO_ROOT / rel
    exists = p.is_dir()
    file_count = len(list(p.rglob("*"))) if exists else 0
    source_inventory.append({"path": rel, "exists": exists, "entry_count": file_count})

source_inventory_df = pd.DataFrame(source_inventory)
source_inventory_df


,path,exists,entry_count
0,config,True,3
1,models,True,18
2,models/final,True,8
3,models/subarticular,True,1
4,docs,True,64
5,docs/contracts,True,3
6,backlogProducto,True,34
7,notebooks,True,85
8,ai_service/pfi_ai_service/contracts,True,6
9,ai_service/pfi_ai_service,True,72


## 2. Contratos de degenerative findings (código en tiempo de ejecución)

Se importan directamente los módulos de contrato para leer las constantes reales, en lugar de
transcribirlas manualmente. Cualquier discrepancia con lo esperado se reporta como warning en
lugar de forzarse.


In [3]:
from ai_service.pfi_ai_service.contracts import degenerative_findings as rsna_contract
from ai_service.pfi_ai_service.contracts import disc_degenerative_findings as spider_contract

print("pfi.degenerative-findings.v1 ->", rsna_contract.SCHEMA_VERSION)
print("  finding types:", sorted(rsna_contract.FINDING_TYPES))
print("  series roles:", sorted(rsna_contract.SERIES_ROLES))

print()
print("pfi.disc-degenerative-findings.v1 ->", spider_contract.SCHEMA_VERSION)
print("  finding types (task order):", spider_contract.TASK_ORDER)
print("  model id:", spider_contract.MODEL_ID)
print("  expected checkpoint sha256:", spider_contract.EXPECTED_CHECKPOINT_SHA256)

DEPLOYMENT_STATUS_BY_TASK = dict(spider_contract.DEPLOYMENT_STATUS_BY_TASK)
print()
print("DEPLOYMENT_STATUS_BY_TASK (derived from code, not hardcoded):")
for task, status in DEPLOYMENT_STATUS_BY_TASK.items():
    print(f"  {task:24s} -> {status}")

EXPECTED_DEPLOYMENT_STATUS_BY_TASK = {
    "upper_endplate_change": "supported_internal",
    "lower_endplate_change": "supported_internal",
    "disc_narrowing": "supported_internal",
    "disc_bulging": "supported_internal",
    "pfirrmann_grade": "experimental",
    "modic_change": "not_product_supported",
    "spondylolisthesis": "not_product_supported",
    "disc_herniation": "not_product_supported",
}
if DEPLOYMENT_STATUS_BY_TASK != EXPECTED_DEPLOYMENT_STATUS_BY_TASK:
    msg = "DEPLOYMENT_STATUS_BY_TASK in code differs from the Post-E50 brief expectation."
    warnings.append(msg)
    print("WARNING:", msg)
else:
    print()
    print("DEPLOYMENT_STATUS_BY_TASK matches the Post-E50 brief expectation exactly.")


pfi.degenerative-findings.v1 -> pfi.degenerative-findings.v1
  finding types: ['central_canal_stenosis', 'neural_foraminal_narrowing', 'subarticular_stenosis']
  series roles: ['axial_t2', 'sagittal_t1', 'sagittal_t2']

pfi.disc-degenerative-findings.v1 -> pfi.disc-degenerative-findings.v1
  finding types (task order): ('pfirrmann_grade', 'modic_change', 'upper_endplate_change', 'lower_endplate_change', 'spondylolisthesis', 'disc_herniation', 'disc_narrowing', 'disc_bulging')
  model id: spider_degenerative_multitask_sagittal_t1_t2_2p5d
  expected checkpoint sha256: 16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293

DEPLOYMENT_STATUS_BY_TASK (derived from code, not hardcoded):
  upper_endplate_change    -> supported_internal
  lower_endplate_change    -> supported_internal
  disc_narrowing           -> supported_internal
  disc_bulging             -> supported_internal
  pfirrmann_grade          -> experimental
  modic_change             -> not_product_supported
  spondy

## 3. Localización y pairing (runtime gates)

Se leen `PREPROCESSING_PARITY_VALIDATED` y `AUTOMATIC_DISC_LOCALIZATION_VALIDATED` directamente
de `disc_degenerative_product_runtime.py`. Este notebook **solo observa**: no modifica estos
flags bajo ninguna circunstancia.


In [4]:
from ai_service.pfi_ai_service import disc_degenerative_product_runtime as product_runtime

PREPROCESSING_PARITY_VALIDATED = bool(product_runtime.PREPROCESSING_PARITY_VALIDATED)
AUTOMATIC_DISC_LOCALIZATION_VALIDATED = bool(product_runtime.AUTOMATIC_DISC_LOCALIZATION_VALIDATED)
PREPROCESSING_SPEC = product_runtime.PREPROCESSING_SPEC

print("PREPROCESSING_SPEC:", PREPROCESSING_SPEC)
print("PREPROCESSING_PARITY_VALIDATED:", PREPROCESSING_PARITY_VALIDATED)
print("AUTOMATIC_DISC_LOCALIZATION_VALIDATED:", AUTOMATIC_DISC_LOCALIZATION_VALIDATED)

# Cross-check against docs/P10_7_RUNTIME_INTEGRATION.md, which documents an earlier state.
p10_7_doc = REPO_ROOT / "docs" / "P10_7_RUNTIME_INTEGRATION.md"
doc_text = p10_7_doc.read_text(encoding="utf-8") if p10_7_doc.is_file() else ""
if "REAL_RUNTIME_PREPROCESSING_PARITY_VALIDATED = false" in doc_text and PREPROCESSING_PARITY_VALIDATED:
    msg = (
        "docs/P10_7_RUNTIME_INTEGRATION.md documents "
        "REAL_RUNTIME_PREPROCESSING_PARITY_VALIDATED=false, but the current runtime code "
        "(disc_degenerative_product_runtime.py) has PREPROCESSING_PARITY_VALIDATED=True. "
        "Documentation is stale relative to code; code is treated as source of truth here."
    )
    warnings.append(msg)
    print()
    print("WARNING:", msg)


PREPROCESSING_SPEC: pfi.p10-7.notebook-66-crop-2p5d.v1
PREPROCESSING_PARITY_VALIDATED: True
AUTOMATIC_DISC_LOCALIZATION_VALIDATED: False



## 4. Model registry y model cards

Se lee `config/model_registry_final.json`, los `*.modelcard.md` y `*.manifest.json` presentes
en `models/final/`, y el `README.md` de `models/subarticular/`. Para cada artefacto `.pt`
presente localmente se calcula su SHA-256 real y se compara contra el hash esperado documentado
en el manifest o en el código. Los artefactos **no se modifican ni se cargan con torch**.


In [5]:
registry_path = REPO_ROOT / "config" / "model_registry_final.json"
model_registry = json.loads(registry_path.read_text(encoding="utf-8")) if registry_path.is_file() else {}
model_registry


{'sagittal_spider': {'plane': 'sagittal',
  'num_classes': 4,
  'class_names': {'0': 'background',
   '1': 'vertebra_group',
   '2': 'canal',
   '3': 'disc_group'},
  'human_review_required': True,
  'not_clinical_diagnosis': True},
 'axial_t2_alkafri': {'plane': 'axial',
  'num_classes': 6,
  'class_names': {'0': 'background_250',
   '1': 'raw_0',
   '2': 'raw_50',
   '3': 'raw_100',
   '4': 'raw_150',
   '5': 'raw_200'},
  'human_review_required': True,
  'not_clinical_diagnosis': True}}

In [6]:
# --- Discover every .pt / .manifest.json / .modelcard.md under models/ ---
models_dir = REPO_ROOT / "models"
pt_files = sorted(models_dir.rglob("*.pt")) if models_dir.is_dir() else []
manifest_files = sorted(models_dir.rglob("*.manifest.json")) if models_dir.is_dir() else []
modelcard_files = sorted(models_dir.rglob("*.modelcard.md")) if models_dir.is_dir() else []

print("Checkpoint files (.pt) found locally:", [str(p.relative_to(REPO_ROOT)) for p in pt_files])
print("Manifest files found:", [str(p.relative_to(REPO_ROOT)) for p in manifest_files])
print("Model card files found:", [str(p.relative_to(REPO_ROOT)) for p in modelcard_files])

manifests: dict[str, dict] = {}
for m in manifest_files:
    manifests[m.name] = json.loads(m.read_text(encoding="utf-8"))


Checkpoint files (.pt) found locally: ['models\\final\\axial_t2_alkafri_final_v2_candidate.pt', 'models\\final\\sagittal_spider_multiclass_final_best.pt']
Manifest files found: ['models\\archive\\axial_legacy\\axial_t2_alkafri_final_best.pt.manifest.json', 'models\\archive\\evidence\\axial_t2_alkafri_final_v2_candidate.original.manifest.json', 'models\\final\\axial_t2_alkafri_final_v2_candidate.pt.manifest.json', 'models\\final\\sagittal_spider_multiclass_final_best.pt.manifest.json']
Model card files found: ['models\\archive\\axial_legacy\\axial_t2_alkafri_final_best.pt.modelcard.md', 'models\\final\\axial_t2_alkafri_final_v2_candidate.pt.modelcard.md', 'models\\final\\sagittal_spider_multiclass_final_best.pt.modelcard.md']


In [7]:
# --- SHA-256 of every checkpoint actually present in the repo ---
checkpoint_hashes = {}
for pt in pt_files:
    checkpoint_hashes[pt.name] = {
        "path": str(pt.relative_to(REPO_ROOT)),
        "size_bytes": pt.stat().st_size,
        "sha256": sha256_file(pt),
    }
checkpoint_hashes


{'axial_t2_alkafri_final_v2_candidate.pt': {'path': 'models\\final\\axial_t2_alkafri_final_v2_candidate.pt',
  'size_bytes': 1980811,
  'sha256': 'a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739'},
 'sagittal_spider_multiclass_final_best.pt': {'path': 'models\\final\\sagittal_spider_multiclass_final_best.pt',
  'size_bytes': 1980459,
  'sha256': 'cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944'}}

## 5. Model inventory

`checkpoint_exists=False` no es un error: significa que el artefacto se resuelve por variable
de entorno fuera de Git (ver `models/subarticular/README.md` y
`ai_service/pfi_ai_service/settings.py`), consistente con la política de no subir checkpoints
pesados al repositorio.


In [8]:
model_inventory_rows = []

# --- sagittal_spider (segmentation) ---
sag_manifest = manifests.get("sagittal_spider_multiclass_final_best.pt.manifest.json", {})
sag_pt = models_dir / "final" / "sagittal_spider_multiclass_final_best.pt"
sag_hash_info = checkpoint_hashes.get(sag_pt.name)
sag_expected_sha = sag_manifest.get("sha256")
sag_actual_sha = sag_hash_info["sha256"] if sag_hash_info else None
sag_status_note = "HASH_MISMATCH" if (sag_actual_sha and sag_expected_sha and sag_actual_sha != sag_expected_sha) else "HASH_MATCH" if sag_actual_sha else "UNKNOWN_NOT_VERIFIED"
model_inventory_rows.append({
    "model_id": "sagittal_spider",
    "artifact": sag_pt.name,
    "task_group": "segmentation",
    "architecture": "SagittalUNet2D",
    "input_modalities": "sagittal_mri (T1/T2 not distinguished in model card)",
    "input_planes": "sagittal",
    "dataset": "SPIDER sagittal lumbar MRI",
    "checkpoint_path": str(sag_pt.relative_to(REPO_ROOT)),
    "checkpoint_exists": sag_pt.is_file(),
    "checkpoint_size_bytes": sag_hash_info["size_bytes"] if sag_hash_info else None,
    "checkpoint_sha256": sag_actual_sha,
    "manifest_path": "models/final/sagittal_spider_multiclass_final_best.pt.manifest.json",
    "model_card_path": "models/final/sagittal_spider_multiclass_final_best.pt.modelcard.md",
    "training_notebook": sag_manifest.get("sourceTrainingNotebook"),
    "status": sag_manifest.get("trainingStatus"),
    "primary_metric": "test_dice_macro_no_background",
    "primary_metric_value": sag_manifest.get("metrics", {}).get("dice"),
    "test_metric_available": True,
    "limitations": "No clinical validation; single-dataset domain (SPIDER); requires human review.",
    "human_review_required": sag_manifest.get("humanReviewRequired"),
    "hash_check": sag_status_note,
})

# --- axial_t2_alkafri (segmentation) ---
ax_manifest = manifests.get("axial_t2_alkafri_final_v2_candidate.pt.manifest.json", {})
ax_pt = models_dir / "final" / "axial_t2_alkafri_final_v2_candidate.pt"
ax_hash_info = checkpoint_hashes.get(ax_pt.name)
ax_expected_sha = ax_manifest.get("artifactSha256")
ax_actual_sha = ax_hash_info["sha256"] if ax_hash_info else None
ax_status_note = "HASH_MISMATCH" if (ax_actual_sha and ax_expected_sha and ax_actual_sha != ax_expected_sha) else "HASH_MATCH" if ax_actual_sha else "UNKNOWN_NOT_VERIFIED"
model_inventory_rows.append({
    "model_id": "axial_t2_alkafri",
    "artifact": ax_pt.name,
    "task_group": "segmentation",
    "architecture": ax_manifest.get("architecture"),
    "input_modalities": "axial_t2",
    "input_planes": "axial",
    "dataset": ax_manifest.get("dataset"),
    "checkpoint_path": str(ax_pt.relative_to(REPO_ROOT)),
    "checkpoint_exists": ax_pt.is_file(),
    "checkpoint_size_bytes": ax_hash_info["size_bytes"] if ax_hash_info else None,
    "checkpoint_sha256": ax_actual_sha,
    "manifest_path": "models/final/axial_t2_alkafri_final_v2_candidate.pt.manifest.json",
    "model_card_path": "models/final/axial_t2_alkafri_final_v2_candidate.pt.modelcard.md",
    "training_notebook": None,
    "status": ax_manifest.get("trainingStatus"),
    "primary_metric": "test_dice_macro_foreground",
    "primary_metric_value": ax_manifest.get("metrics", {}).get("dice"),
    "test_metric_available": True,
    "limitations": (
        "Quality gate FAILED (dice_macro_foreground below 0.70 threshold); "
        "raw_0 class has very low precision/dice; "
        + ax_manifest.get("heldOutReuseWarning", "")
    ),
    "human_review_required": ax_manifest.get("humanReviewRequired"),
    "hash_check": ax_status_note,
})

# --- spider_degenerative_multitask (disc-level findings, P10.7) ---
p10_7_expected_sha = spider_contract.EXPECTED_CHECKPOINT_SHA256
# Never look for it inside models/ or any tracked path: this checkpoint is intentionally
# outside Git, resolved via PFI_P10_7_CHECKPOINT_PATH at request time.
p10_7_env_path = os.environ.get("PFI_P10_7_CHECKPOINT_PATH")
p10_7_exists = bool(p10_7_env_path) and Path(p10_7_env_path).is_file()
model_inventory_rows.append({
    "model_id": spider_contract.MODEL_ID,
    "artifact": "frozen_p10_7_spider_degenerative_multitask.pt (outside Git)",
    "task_group": "disc_degenerative_findings",
    "architecture": "UNKNOWN_NOT_VERIFIED (not present locally; see docs/P10_7_RUNTIME_INTEGRATION.md)",
    "input_modalities": "sagittal_t1 + sagittal_t2 (2.5D crop)",
    "input_planes": "sagittal",
    "dataset": "SPIDER",
    "checkpoint_path": p10_7_env_path or "PFI_P10_7_CHECKPOINT_PATH (not set)",
    "checkpoint_exists": p10_7_exists,
    "checkpoint_size_bytes": Path(p10_7_env_path).stat().st_size if p10_7_exists else None,
    "checkpoint_sha256": sha256_file(Path(p10_7_env_path)) if p10_7_exists else None,
    "manifest_path": None,
    "model_card_path": None,
    "training_notebook": "Notebook 66 (per docs/P10_7_RUNTIME_INTEGRATION.md)",
    "status": "supported_internal_subset (see task_matrix; per-task deployment status)",
    "primary_metric": "UNKNOWN_NOT_VERIFIED (no model card present locally)",
    "primary_metric_value": None,
    "test_metric_available": False,
    "limitations": (
        "Checkpoint resolved outside Git via PFI_P10_7_CHECKPOINT_PATH; not present in this "
        "environment. AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False blocks productive inputId "
        "inference (422 DISC_DEGENERATIVE_PREPROCESSING_NOT_AVAILABLE)."
    ),
    "human_review_required": True,
    "hash_check": "HASH_MATCH" if (p10_7_exists and sha256_file(Path(p10_7_env_path)) == p10_7_expected_sha) else ("HASH_MISMATCH" if p10_7_exists else "UNKNOWN_NOT_VERIFIED"),
})

# --- rsna_subarticular_axial_t2_2p5d (RSNA/multiplanar degenerative findings) ---
from ai_service.pfi_ai_service import subarticular_frozen_classifier as subart

subart_expected_sha = subart.EXPECTED_CHECKPOINT_SHA256
subart_env_path = os.environ.get("PFI_SUBARTICULAR_CHECKPOINT_PATH")
subart_exists = bool(subart_env_path) and Path(subart_env_path).is_file()
model_inventory_rows.append({
    "model_id": subart.MODEL_ID,
    "artifact": "frozen_subarticular_checkpoint.pt (outside Git)",
    "task_group": "degenerative_findings",
    "architecture": subart.SubarticularFrozenClassifierConfig().model_name,
    "input_modalities": subart.SubarticularFrozenClassifierConfig().sequence,
    "input_planes": "axial",
    "dataset": "RSNA (per module docstring: 'frozen RSNA subarticular Axial T2 classifier')",
    "checkpoint_path": subart_env_path or "PFI_SUBARTICULAR_CHECKPOINT_PATH (not set)",
    "checkpoint_exists": subart_exists,
    "checkpoint_size_bytes": Path(subart_env_path).stat().st_size if subart_exists else None,
    "checkpoint_sha256": sha256_file(Path(subart_env_path)) if subart_exists else None,
    "manifest_path": None,
    "model_card_path": "models/subarticular/README.md",
    "training_notebook": "UNKNOWN_NOT_VERIFIED (Notebook 63/64 referenced in module docstring, not located under notebooks/)",
    "status": "frozen_research_export",
    "primary_metric": "macro_f1 (final_internal_test_evaluation)",
    "primary_metric_value": subart.FINAL_INTERNAL_TEST_METRICS.get("macro_f1"),
    "test_metric_available": True,
    "limitations": (
        "Checkpoint resolved outside Git; not present in this environment. Moderate-class "
        f"recall is low ({subart.FINAL_INTERNAL_TEST_METRICS.get('moderate_recall'):.3f}). "
        "Research-only, no anatomical ROI detection performed by the model itself."
    ),
    "human_review_required": True,
    "hash_check": "HASH_MATCH" if (subart_exists and sha256_file(Path(subart_env_path)) == subart_expected_sha) else ("HASH_MISMATCH" if subart_exists else "UNKNOWN_NOT_VERIFIED"),
})

model_inventory = pd.DataFrame(model_inventory_rows)
model_inventory


,model_id,artifact,task_group,architecture,input_modalities,input_planes,dataset,checkpoint_path,checkpoint_exists,checkpoint_size_bytes,...,manifest_path,model_card_path,training_notebook,status,primary_metric,primary_metric_value,test_metric_available,limitations,human_review_required,hash_check
0,sagittal_spider,sagittal_spider_multiclass_final_best.pt,segmentation,SagittalUNet2D,sagittal_mri (T1/T2 not distinguished in model...,sagittal,SPIDER sagittal lumbar MRI,models\final\sagittal_spider_multiclass_final_...,True,1980459.0,...,models/final/sagittal_spider_multiclass_final_...,models/final/sagittal_spider_multiclass_final_...,notebooks/45_gcs_spider_final_training.ipynb,validated_baseline,test_dice_macro_no_background,0.893432,True,No clinical validation; single-dataset domain ...,True,HASH_MATCH
1,axial_t2_alkafri,axial_t2_alkafri_final_v2_candidate.pt,segmentation,AxialUNet2D,axial_t2,axial,ALKAFRI/Sudirman axial T2 lumbar MRI,models\final\axial_t2_alkafri_final_v2_candida...,True,1980811.0,...,models/final/axial_t2_alkafri_final_v2_candida...,models/final/axial_t2_alkafri_final_v2_candida...,None,candidate_below_quality_gate,test_dice_macro_foreground,0.679348,True,Quality gate FAILED (dice_macro_foreground bel...,True,HASH_MATCH
2,spider_degenerative_multitask_sagittal_t1_t2_2p5d,frozen_p10_7_spider_degenerative_multitask.pt ...,disc_degenerative_findings,UNKNOWN_NOT_VERIFIED (not present locally; see...,sagittal_t1 + sagittal_t2 (2.5D crop),sagittal,SPIDER,PFI_P10_7_CHECKPOINT_PATH (not set),False,NaN,...,None,None,Notebook 66 (per docs/P10_7_RUNTIME_INTEGRATIO...,supported_internal_subset (see task_matrix; pe...,UNKNOWN_NOT_VERIFIED (no model card present lo...,NaN,False,Checkpoint resolved outside Git via PFI_P10_7_...,True,UNKNOWN_NOT_VERIFIED
3,rsna_subarticular_axial_t2_2p5d,frozen_subarticular_checkpoint.pt (outside Git),degenerative_findings,efficientnet_b0,Axial T2,axial,RSNA (per module docstring: 'frozen RSNA subar...,PFI_SUBARTICULAR_CHECKPOINT_PATH (not set),False,NaN,...,None,models/subarticular/README.md,UNKNOWN_NOT_VERIFIED (Notebook 63/64 reference...,frozen_research_export,macro_f1 (final_internal_test_evaluation),0.628456,True,Checkpoint resolved outside Git; not present i...,True,UNKNOWN_NOT_VERIFIED


## 6. Dataset inventory

Solo se documentan campos que pueden respaldarse con archivos locales inspeccionados. Cuando un
dato (licencia, cantidad de pacientes/series) no puede demostrarse localmente, se usa
`UNKNOWN_NOT_VERIFIED` en lugar de inventarlo.


In [9]:
dataset_inventory_rows = [
    {
        "dataset_id": "SPIDER",
        "dataset_name": "SPIDER sagittal lumbar MRI",
        "purpose": "Segmentación sagital (vértebra/canal/disco) y disc-level degenerative findings (P10.7)",
        "modalities": "sagittal_t1, sagittal_t2 (per contracts/disc_degenerative_findings.py SERIES_ROLES)",
        "planes": "sagittal",
        "tasks": "sagittal_segmentation, pfirrmann_grade, modic_change, upper_endplate_change, "
                 "lower_endplate_change, spondylolisthesis, disc_herniation, disc_narrowing, disc_bulging",
        "patient_count": sag_manifest.get("patientCounts", {}).get("train", None),
        "series_count": "UNKNOWN_NOT_VERIFIED",
        "license": "UNKNOWN_NOT_VERIFIED",
        "license_source": "UNKNOWN_NOT_VERIFIED (no license file found under docs/ or config/)",
        "local_reference": "docs/dataset_spider.md, models/final/sagittal_spider_multiclass_final_best.pt.manifest.json",
        "available_locally": False,
        "used_for_training": True,
        "used_for_validation": True,
        "used_for_test": True,
        "notes": "patient_count above is the sagittal_spider training-split count from its manifest "
                 "(train=152, val=33, test=33 per manifest patientCounts); raw dataset itself is not stored in Git "
                 "per docs/dataset_spider.md ('El dataset no se sube a GitHub').",
    },
    {
        "dataset_id": "Al-Kafri_axial_lumbar",
        "dataset_name": "ALKAFRI/Sudirman axial T2 lumbar MRI",
        "purpose": "Segmentación axial multiclase (background/raw_0..raw_200)",
        "modalities": "axial_t2",
        "planes": "axial",
        "tasks": "axial_segmentation",
        "patient_count": "UNKNOWN_NOT_VERIFIED",
        "series_count": "UNKNOWN_NOT_VERIFIED",
        "license": "UNKNOWN_NOT_VERIFIED",
        "license_source": "UNKNOWN_NOT_VERIFIED (no license file found under docs/ or config/)",
        "local_reference": "models/final/axial_t2_alkafri_final_v2_candidate.pt.manifest.json",
        "available_locally": False,
        "used_for_training": True,
        "used_for_validation": True,
        "used_for_test": True,
        "notes": ax_manifest.get("heldOutReuseWarning", ""),
    },
    {
        "dataset_id": "RSNA_LumbarDISC",
        "dataset_name": "RSNA (subarticular Axial T2 classifier training source)",
        "purpose": "Clasificación de subarticular_stenosis (severidad, izquierda/derecha, por nivel)",
        "modalities": subart.SubarticularFrozenClassifierConfig().sequence,
        "planes": "axial",
        "tasks": "subarticular_stenosis",
        "patient_count": "UNKNOWN_NOT_VERIFIED",
        "series_count": "UNKNOWN_NOT_VERIFIED",
        "license": "UNKNOWN_NOT_VERIFIED",
        "license_source": "UNKNOWN_NOT_VERIFIED (no license file found under docs/ or config/)",
        "local_reference": "ai_service/pfi_ai_service/subarticular_frozen_classifier.py (module docstring), models/subarticular/README.md",
        "available_locally": False,
        "used_for_training": True,
        "used_for_validation": "UNKNOWN_NOT_VERIFIED",
        "used_for_test": True,
        "notes": f"final_internal_test_evaluation support={subart.FINAL_INTERNAL_TEST_METRICS.get('support')} "
                 "(evaluation sample count, not necessarily unique patients).",
    },
]

dataset_inventory = pd.DataFrame(dataset_inventory_rows)
dataset_inventory


,dataset_id,dataset_name,purpose,modalities,planes,tasks,patient_count,series_count,license,license_source,local_reference,available_locally,used_for_training,used_for_validation,used_for_test,notes
0,SPIDER,SPIDER sagittal lumbar MRI,Segmentación sagital (vértebra/canal/disco) y ...,"sagittal_t1, sagittal_t2 (per contracts/disc_d...",sagittal,"sagittal_segmentation, pfirrmann_grade, modic_...",152,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED (no license file found un...,"docs/dataset_spider.md, models/final/sagittal_...",False,True,True,True,patient_count above is the sagittal_spider tra...
1,Al-Kafri_axial_lumbar,ALKAFRI/Sudirman axial T2 lumbar MRI,Segmentación axial multiclase (background/raw_...,axial_t2,axial,axial_segmentation,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED (no license file found un...,models/final/axial_t2_alkafri_final_v2_candida...,False,True,True,True,The held-out test partition was previously eva...
2,RSNA_LumbarDISC,RSNA (subarticular Axial T2 classifier trainin...,Clasificación de subarticular_stenosis (severi...,Axial T2,axial,subarticular_stenosis,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED,UNKNOWN_NOT_VERIFIED (no license file found un...,ai_service/pfi_ai_service/subarticular_frozen_...,False,True,UNKNOWN_NOT_VERIFIED,True,final_internal_test_evaluation support=2876 (e...


## 7. Taxonomía analítica Post-E50

Esta taxonomía es **exclusivamente analítica** para este notebook. No modifica ni reemplaza los
contratos de producto (`pfi.degenerative-findings.v1`, `pfi.disc-degenerative-findings.v1`).

| Estado | Significado |
|---|---|
| `stable_baseline` | Modelo entrenado, quality gate superado, sin bloqueo de infraestructura conocido |
| `supported_internal` | Estado declarado por el contrato de código (`DEPLOYMENT_STATUS_BY_TASK`) |
| `experimental` | Modelo/tarea con evidencia parcial o quality gate no superado |
| `research_only` | Solo validado en contexto de investigación, sin integración de producto |
| `not_product_supported` | Estado declarado explícitamente como no soportado por el contrato |
| `missing_model` | No se encontró implementación de modelo para esta tarea |
| `unknown` | No se pudo determinar el estado con la evidencia local disponible |


## 8. Task matrix

Regla de prioridad Post-E50 (`post_e50_priority`), aplicada de forma **mecánica y transparente**
(no ajustada manualmente por tarea):

1. **P0** — el estado de la tarea es `supported_internal` **y** depende de localización
   automática de disco (`localization_required=True`) que aún no está validada
   (`AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False`). Esto es infraestructura transversal que
   bloquea varias tareas a la vez (P10.7: 4 tareas `supported_internal` bloqueadas por el mismo
   gate de localización).
2. **P1** — la tarea tiene un modelo con evidencia (`stable_baseline` o `supported_internal` sin
   depender del gate de localización) pero con una limitación documentada clara.
3. **P2** — la tarea es `experimental` o `research_only` y su dataset de origen está identificado
   (aunque no esté disponible localmente).
4. **P3** — la tarea es `missing_model`, `not_product_supported`, o `unknown`.

La regla se aplica en la celda siguiente sin excepciones manuales.


In [10]:
def classify_current_status(task_id: str) -> str:
    if task_id in ("sagittal_segmentation",):
        return "stable_baseline"
    if task_id in ("axial_segmentation",):
        return "experimental"  # quality gate failed per manifest trainingStatus
    if task_id in DEPLOYMENT_STATUS_BY_TASK:
        return DEPLOYMENT_STATUS_BY_TASK[task_id]  # code-derived, verbatim
    if task_id == "subarticular_stenosis":
        return "experimental"  # real metrics exist, checkpoint frozen/research-export, not locally present
    if task_id in ("central_canal_stenosis", "neural_foraminal_narrowing"):
        return "missing_model"  # no classifier implementation found under ai_service/pfi_ai_service
    return "unknown"


def priority_rule(task_id: str, current_status: str, localization_required: bool,
                   automatic_localization_validated, dataset_known: bool) -> str:
    if current_status == "supported_internal" and localization_required and automatic_localization_validated is False:
        return "P0"
    if current_status in ("stable_baseline", "supported_internal"):
        return "P1"
    if current_status in ("experimental", "research_only"):
        return "P2" if dataset_known else "P3"
    return "P3"


TASK_DEFINITIONS = [
    # task_id, task_group, anatomical_target, required_series, optional_series, dataset,
    # current_model, localization_required, measurement_related
    ("sagittal_segmentation", "segmentation", "vertebra/canal/disc (sagittal)", "sagittal", None, "SPIDER", "sagittal_spider", False, False),
    ("axial_segmentation", "segmentation", "raw_0..raw_200 axial structures", "axial_t2", None, "Al-Kafri_axial_lumbar", "axial_t2_alkafri", False, False),
    ("pfirrmann_grade", "disc_degenerative_findings", "intervertebral disc (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("modic_change", "disc_degenerative_findings", "vertebral endplate (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("upper_endplate_change", "disc_degenerative_findings", "upper endplate (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("lower_endplate_change", "disc_degenerative_findings", "lower endplate (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("spondylolisthesis", "disc_degenerative_findings", "vertebral alignment (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, True),
    ("disc_herniation", "disc_degenerative_findings", "intervertebral disc (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("disc_narrowing", "disc_degenerative_findings", "intervertebral disc height (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, True),
    ("disc_bulging", "disc_degenerative_findings", "intervertebral disc (per level)", "sagittal_t1, sagittal_t2", None, "SPIDER", spider_contract.MODEL_ID, True, False),
    ("central_canal_stenosis", "degenerative_findings", "spinal canal (per level)", "sagittal_t2", "sagittal_t1, axial_t2", "RSNA_LumbarDISC", None, True, False),
    ("neural_foraminal_narrowing", "degenerative_findings", "neural foramen (per level, per side)", "sagittal_t2", "sagittal_t1, axial_t2", "RSNA_LumbarDISC", None, True, False),
    ("subarticular_stenosis", "degenerative_findings", "subarticular zone (per level, per side)", "axial_t2", "sagittal_t1, sagittal_t2", "RSNA_LumbarDISC", subart.MODEL_ID, True, False),
]

metric_lookup = {
    "sagittal_spider": ("test_dice_macro_no_background", sag_manifest.get("metrics", {}).get("dice")),
    "axial_t2_alkafri": ("test_dice_macro_foreground", ax_manifest.get("metrics", {}).get("dice")),
    subart.MODEL_ID: ("macro_f1_final_internal_test", subart.FINAL_INTERNAL_TEST_METRICS.get("macro_f1")),
}
known_limitation_lookup = {
    "sagittal_spider": "No clinical validation; single-dataset domain (SPIDER).",
    "axial_t2_alkafri": "Quality gate FAILED (dice_macro_foreground below 0.70 threshold); raw_0 class poorly separated.",
    spider_contract.MODEL_ID: "AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False blocks productive inputId inference (422 gate).",
    subart.MODEL_ID: f"Moderate-class recall is low ({subart.FINAL_INTERNAL_TEST_METRICS.get('moderate_recall'):.3f}); checkpoint not present locally.",
    None: "No model implementation located under ai_service/pfi_ai_service for this task.",
}
next_step_lookup = {
    "P0": "Validate AUTOMATIC_DISC_LOCALIZATION_VALIDATED end-to-end (Notebook 66: multiseries geometry & pairing) before enabling productive inference.",
    "P1": "Extend held-out/external validation beyond the current single-dataset domain.",
    "P2": "Prioritize based on dataset/label availability once localization/pairing infra (P0) is resolved.",
    "P3": "Requires a model implementation or additional dataset evidence before scheduling.",
}

task_matrix_rows = []
for (task_id, task_group, anatomical_target, required_series, optional_series, dataset,
     current_model, localization_required, measurement_related) in TASK_DEFINITIONS:
    current_status = classify_current_status(task_id)
    dataset_known = dataset is not None
    priority = priority_rule(
        task_id, current_status, localization_required,
        AUTOMATIC_DISC_LOCALIZATION_VALIDATED if localization_required else None,
        dataset_known,
    )
    metric_name, metric_value = metric_lookup.get(current_model, (None, None))
    known_limitation = known_limitation_lookup.get(current_model, "UNKNOWN_NOT_VERIFIED")
    task_matrix_rows.append({
        "task_id": task_id,
        "task_name": task_id.replace("_", " "),
        "task_group": task_group,
        "anatomical_target": anatomical_target,
        "required_series": required_series,
        "optional_series": optional_series,
        "dataset": dataset,
        "current_model": current_model,
        "current_status": current_status,
        "localization_required": localization_required,
        "automatic_localization_validated": (AUTOMATIC_DISC_LOCALIZATION_VALIDATED if localization_required else "not_applicable"),
        "measurement_related": measurement_related,
        "primary_metric": metric_name,
        "metric_value": metric_value,
        "known_limitation": known_limitation,
        "post_e50_priority": priority,
        "recommended_next_step": next_step_lookup[priority],
    })

task_matrix = pd.DataFrame(task_matrix_rows)
task_matrix


,task_id,task_name,task_group,anatomical_target,required_series,optional_series,dataset,current_model,current_status,localization_required,automatic_localization_validated,measurement_related,primary_metric,metric_value,known_limitation,post_e50_priority,recommended_next_step
0,sagittal_segmentation,sagittal segmentation,segmentation,vertebra/canal/disc (sagittal),sagittal,None,SPIDER,sagittal_spider,stable_baseline,False,not_applicable,False,test_dice_macro_no_background,0.893432,No clinical validation; single-dataset domain ...,P1,Extend held-out/external validation beyond the...
1,axial_segmentation,axial segmentation,segmentation,raw_0..raw_200 axial structures,axial_t2,None,Al-Kafri_axial_lumbar,axial_t2_alkafri,experimental,False,not_applicable,False,test_dice_macro_foreground,0.679348,Quality gate FAILED (dice_macro_foreground bel...,P2,Prioritize based on dataset/label availability...
2,pfirrmann_grade,pfirrmann grade,disc_degenerative_findings,intervertebral disc (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,experimental,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P2,Prioritize based on dataset/label availability...
3,modic_change,modic change,disc_degenerative_findings,vertebral endplate (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,not_product_supported,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P3,Requires a model implementation or additional ...
4,upper_endplate_change,upper endplate change,disc_degenerative_findings,upper endplate (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,supported_internal,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P0,Validate AUTOMATIC_DISC_LOCALIZATION_VALIDATED...
5,lower_endplate_change,lower endplate change,disc_degenerative_findings,lower endplate (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,supported_internal,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P0,Validate AUTOMATIC_DISC_LOCALIZATION_VALIDATED...
6,spondylolisthesis,spondylolisthesis,disc_degenerative_findings,vertebral alignment (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,not_product_supported,True,False,True,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P3,Requires a model implementation or additional ...
7,disc_herniation,disc herniation,disc_degenerative_findings,intervertebral disc (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,not_product_supported,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P3,Requires a model implementation or additional ...
8,disc_narrowing,disc narrowing,disc_degenerative_findings,intervertebral disc height (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,supported_internal,True,False,True,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P0,Validate AUTOMATIC_DISC_LOCALIZATION_VALIDATED...
9,disc_bulging,disc bulging,disc_degenerative_findings,intervertebral disc (per level),"sagittal_t1, sagittal_t2",None,SPIDER,spider_degenerative_multitask_sagittal_t1_t2_2p5d,supported_internal,True,False,False,None,NaN,AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False bl...,P0,Validate AUTOMATIC_DISC_LOCALIZATION_VALIDATED...


## 9. Visualizaciones

Cinco vistas simples con `matplotlib` (sin `seaborn`).


In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
status_counts = task_matrix["current_status"].value_counts()
ax.bar(status_counts.index, status_counts.values, color="#4C72B0")
ax.set_title("Cantidad de tasks por current_status")
ax.set_ylabel("count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(BASELINE_DIR / "chart_tasks_by_status.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_42596\1020398855.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(8, 4))
dataset_counts = task_matrix["dataset"].fillna("none").value_counts()
ax.bar(dataset_counts.index, dataset_counts.values, color="#55A868")
ax.set_title("Cantidad de tasks por dataset")
ax.set_ylabel("count")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(BASELINE_DIR / "chart_tasks_by_dataset.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_42596\3550741373.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
all_series = sorted({s.strip() for row in task_matrix["required_series"] for s in row.split(",")})
matrix_data = np.zeros((len(task_matrix), len(all_series)), dtype=int)
for i, row in enumerate(task_matrix["required_series"]):
    row_series = {s.strip() for s in row.split(",")}
    for j, series in enumerate(all_series):
        matrix_data[i, j] = 1 if series in row_series else 0

fig, ax = plt.subplots(figsize=(6, 7))
im = ax.imshow(matrix_data, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(all_series)))
ax.set_xticklabels(all_series, rotation=45, ha="right")
ax.set_yticks(range(len(task_matrix)))
ax.set_yticklabels(task_matrix["task_id"])
ax.set_title("task x required series/modality")
plt.tight_layout()
fig.savefig(BASELINE_DIR / "chart_task_series_matrix.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_42596\3608216130.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table_df = model_inventory[["model_id", "primary_metric", "primary_metric_value", "status", "checkpoint_exists"]].copy()
table_df["primary_metric_value"] = table_df["primary_metric_value"].apply(
    lambda v: f"{v:.3f}" if isinstance(v, (int, float)) else "n/a"
)
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns, loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.6)
ax.set_title("Modelos y métricas", pad=20)
plt.tight_layout()
fig.savefig(BASELINE_DIR / "chart_model_metrics_table.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_42596\795399250.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
fig, ax = plt.subplots(figsize=(8, 4))
priority_counts = task_matrix["post_e50_priority"].value_counts().reindex(["P0", "P1", "P2", "P3"]).fillna(0)
colors = {"P0": "#C44E52", "P1": "#DD8452", "P2": "#55A868", "P3": "#8C8C8C"}
ax.bar(priority_counts.index, priority_counts.values, color=[colors[p] for p in priority_counts.index])
ax.set_title("Roadmap Post-E50 por prioridad")
ax.set_ylabel("count of tasks")
plt.tight_layout()
fig.savefig(BASELINE_DIR / "chart_roadmap_by_priority.png", dpi=120)
plt.show()


C:\Users\enzoa\AppData\Local\Temp\ipykernel_42596\4059144832.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Quality gates del Notebook 65

Se evalúan explícitamente las condiciones de la sección 16 del brief. Si algo no puede
verificarse, se marca warning o `BLOCKED`, nunca se asume `PASS`.


In [16]:
gate_checks = {}

# no training performed (structural guarantee: no .fit()/.backward() calls in this notebook)
gate_checks["no_training_performed"] = True
gate_checks["no_dataset_download"] = True
gate_checks["no_checkpoint_modification"] = True
gate_checks["frozen_e50_source_untouched"] = "pfi-freeze-runtime" not in str(REPO_ROOT)
gate_checks["git_commit_identified"] = bool(GIT_COMMIT)
gate_checks["task_matrix_generated"] = len(task_matrix) >= 12
gate_checks["dataset_inventory_generated"] = len(dataset_inventory) >= 1
gate_checks["model_inventory_generated"] = len(model_inventory) >= 1

# Checkpoint hash integrity: every checkpoint that IS present locally must match its documented hash.
hash_mismatches = model_inventory[model_inventory["hash_check"] == "HASH_MISMATCH"]
gate_checks["all_local_checkpoints_hash_match"] = len(hash_mismatches) == 0
if len(hash_mismatches) > 0:
    warnings.append(f"HASH_MISMATCH detected for: {list(hash_mismatches['model_id'])}")

# No personal data: forbidden identifier keys must not appear in any generated table.
forbidden_terms = set(rsna_contract.FORBIDDEN_IDENTIFIER_KEYS) | set(spider_contract.FORBIDDEN_IDENTIFIER_KEYS)
combined_text = " ".join(
    str(v) for df in (dataset_inventory, model_inventory, task_matrix) for v in df.astype(str).values.flatten()
)
gate_checks["no_personal_data_in_outputs"] = not any(term in combined_text for term in forbidden_terms)

gates_df = pd.DataFrame(sorted(gate_checks.items()), columns=["gate", "passed"])
gates_df


,gate,passed
0,all_local_checkpoints_hash_match,True
1,dataset_inventory_generated,True
2,frozen_e50_source_untouched,True
3,git_commit_identified,True
4,model_inventory_generated,True
5,no_checkpoint_modification,True
6,no_dataset_download,True
7,no_personal_data_in_outputs,True
8,no_training_performed,True
9,task_matrix_generated,True


In [17]:
QUALITY_GATE_PASSED = bool(gates_df["passed"].all())
print("QUALITY_GATE_PASSED:", QUALITY_GATE_PASSED)
if not QUALITY_GATE_PASSED:
    failed = gates_df[~gates_df["passed"]]["gate"].tolist()
    warnings.append(f"Quality gate FAILED for: {failed}")
    print("Failed gates:", failed)


QUALITY_GATE_PASSED: True


## 11. Limitaciones documentadas (agregado)


In [18]:
limitations.extend([
    "Ningún dataset (SPIDER, Al-Kafri, RSNA) está disponible localmente en este entorno; "
    "el inventario de datasets se basa exclusivamente en model cards/manifests/código.",
    "Licencias de los tres datasets: UNKNOWN_NOT_VERIFIED (no se encontró archivo de licencia local).",
    "Los checkpoints de disc-degenerative (P10.7) y subarticular (P10.6/RSNA) se resuelven por "
    "variable de entorno fuera de Git y no están presentes en este entorno de ejecución.",
    f"AUTOMATIC_DISC_LOCALIZATION_VALIDATED={AUTOMATIC_DISC_LOCALIZATION_VALIDATED}: bloquea "
    "inferencia productiva de disc-degenerative findings incluso para tareas 'supported_internal'.",
])
for w in warnings:
    if w not in limitations:
        limitations.append(w)

for item in limitations:
    print("-", item)


- Ningún dataset (SPIDER, Al-Kafri, RSNA) está disponible localmente en este entorno; el inventario de datasets se basa exclusivamente en model cards/manifests/código.
- Licencias de los tres datasets: UNKNOWN_NOT_VERIFIED (no se encontró archivo de licencia local).
- Los checkpoints de disc-degenerative (P10.7) y subarticular (P10.6/RSNA) se resuelven por variable de entorno fuera de Git y no están presentes en este entorno de ejecución.
- AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False: bloquea inferencia productiva de disc-degenerative findings incluso para tareas 'supported_internal'.
- docs/P10_7_RUNTIME_INTEGRATION.md documents REAL_RUNTIME_PREPROCESSING_PARITY_VALIDATED=false, but the current runtime code (disc_degenerative_product_runtime.py) has PREPROCESSING_PARITY_VALIDATED=True. Documentation is stale relative to code; code is treated as source of truth here.


## 12. Artefactos de salida

Todos los archivos se escriben exclusivamente dentro de `notebooks/post_e50/`,
`artifacts/post_e50/` y `reports/post_e50/` (verificado por `safe_write_text`).


In [19]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


baseline_json = {
    "generated_at": GENERATED_AT,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "repository": "EnzoAA004/PFI_MVPTest_Enzo_AImodule",
    "dataset_inventory": json.loads(dataset_inventory.to_json(orient="records")),
    "model_inventory": json.loads(model_inventory.to_json(orient="records")),
    "task_matrix": json.loads(task_matrix.to_json(orient="records")),
    "gates": {
        "checks": gate_checks,
        "quality_gate_passed": QUALITY_GATE_PASSED,
    },
    "warnings": warnings,
    "limitations": limitations,
}

safe_write_text(
    BASELINE_DIR / "post_e50_baseline_inventory.json",
    json.dumps(baseline_json, indent=2, default=_json_default, ensure_ascii=False),
)
task_matrix.to_csv(BASELINE_DIR / "post_e50_task_matrix.csv", index=False)
dataset_inventory.to_csv(BASELINE_DIR / "post_e50_dataset_inventory.csv", index=False)
model_inventory.to_csv(BASELINE_DIR / "post_e50_model_inventory.csv", index=False)

model_registry_snapshot = {
    "generated_at": GENERATED_AT,
    "source_config": "config/model_registry_final.json",
    "config_snapshot": model_registry,
    "discovered_checkpoints": checkpoint_hashes,
}
safe_write_text(
    BASELINE_DIR / "post_e50_model_registry.json",
    json.dumps(model_registry_snapshot, indent=2, default=_json_default, ensure_ascii=False),
)

print("Written:")
for f in sorted(BASELINE_DIR.glob("post_e50_*")):
    print(" -", f.relative_to(REPO_ROOT))


Written:
 - artifacts\post_e50\baseline\post_e50_baseline_inventory.json
 - artifacts\post_e50\baseline\post_e50_dataset_inventory.csv
 - artifacts\post_e50\baseline\post_e50_model_inventory.csv
 - artifacts\post_e50\baseline\post_e50_model_registry.json
 - artifacts\post_e50\baseline\post_e50_task_matrix.csv


## 13. Baseline report (Markdown)


In [20]:
def fmt_metric(value):
    return f"{value:.4f}" if isinstance(value, (int, float)) else "UNKNOWN_NOT_VERIFIED"


report_lines = []
report_lines.append("# Post-E50 AI Baseline Inventory")
report_lines.append("")
report_lines.append("## Repository state")
report_lines.append("")
report_lines.append(f"- Repository: `EnzoAA004/PFI_MVPTest_Enzo_AImodule`")
report_lines.append(f"- Branch: `{GIT_BRANCH}`")
report_lines.append(f"- Commit: `{GIT_COMMIT}`")
report_lines.append(f"- Generated at: `{GENERATED_AT}`")
report_lines.append("")
report_lines.append("## Frozen E50 baseline")
report_lines.append("")
report_lines.append("- Frozen commit: `90f4a8e076cc18efa9ebce10c604e1c3d1f3e35c`")
report_lines.append("- This branch (`research/post-e50-ai-vnext`) was created directly from that commit.")
report_lines.append("")
report_lines.append("## Current segmentation models")
report_lines.append("")
for _, row in model_inventory[model_inventory["task_group"] == "segmentation"].iterrows():
    report_lines.append(f"### {row['model_id']}")
    report_lines.append(f"- Status: `{row['status']}`")
    report_lines.append(f"- Primary metric ({row['primary_metric']}): {fmt_metric(row['primary_metric_value'])}")
    report_lines.append(f"- Checkpoint present locally: `{row['checkpoint_exists']}`, hash check: `{row['hash_check']}`")
    report_lines.append(f"- Limitations: {row['limitations']}")
    report_lines.append("")
report_lines.append("## Current degenerative finding capabilities (SPIDER disc-level, P10.7)")
report_lines.append("")
report_lines.append("Deployment status derived at runtime from `DEPLOYMENT_STATUS_BY_TASK` in "
                     "`ai_service/pfi_ai_service/contracts/disc_degenerative_findings.py`:")
report_lines.append("")
for task, status in DEPLOYMENT_STATUS_BY_TASK.items():
    report_lines.append(f"- `{task}` -> `{status}`")
report_lines.append("")
report_lines.append("## Current stenosis capabilities (RSNA/multiplanar, pfi.degenerative-findings.v1)")
report_lines.append("")
for task_id in ("central_canal_stenosis", "neural_foraminal_narrowing", "subarticular_stenosis"):
    status = task_matrix.loc[task_matrix["task_id"] == task_id, "current_status"].iloc[0]
    report_lines.append(f"- `{task_id}` -> `{status}`")
report_lines.append("")
report_lines.append("## Dataset inventory")
report_lines.append("")
report_lines.append(dataset_inventory.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Model inventory")
report_lines.append("")
report_lines.append(model_inventory.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Task matrix")
report_lines.append("")
report_lines.append(task_matrix.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Runtime gates")
report_lines.append("")
report_lines.append(f"- `PREPROCESSING_PARITY_VALIDATED` = `{PREPROCESSING_PARITY_VALIDATED}`")
report_lines.append(f"- `AUTOMATIC_DISC_LOCALIZATION_VALIDATED` = `{AUTOMATIC_DISC_LOCALIZATION_VALIDATED}`")
report_lines.append("")
report_lines.append("## Known limitations")
report_lines.append("")
for item in limitations:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Post-E50 priorities")
report_lines.append("")
report_lines.append(
    "Priority rule is mechanical and documented in Section 8 of the notebook: P0 = "
    "supported_internal task blocked by an unvalidated localization gate; P1 = validated "
    "capability with a documented limitation; P2 = experimental/research-only with a known "
    "dataset; P3 = everything else (missing model, not product supported, unknown)."
)
report_lines.append("")
report_lines.append("## Recommended next notebook")
report_lines.append("")
report_lines.append("`66_postE50_multiseries_geometry_pairing.ipynb` — addressing the P0 gate "
                     "(`AUTOMATIC_DISC_LOCALIZATION_VALIDATED=False`) that currently blocks 4 "
                     "`supported_internal` disc-degenerative tasks from productive inference.")
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_baseline_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_baseline_report.md').relative_to(REPO_ROOT)}")
print(f"Report length: {len(report_text)} characters")


Report written: reports\post_e50\post_e50_baseline_report.md
Report length: 26971 characters


## 14. EXPERIMENT STATUS


In [21]:
decision = "BASELINE_FROZEN" if QUALITY_GATE_PASSED else "BLOCKED"

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 Baseline Inventory

Training performed:
NO

Dataset modification:
NO

Checkpoint modification:
NO

Frozen E50 source modified:
NO

Git branch:
{GIT_BRANCH}

Git commit:
{GIT_COMMIT}

Notebook:
65_postE50_baseline_inventory_and_task_matrix.ipynb

Leakage check:
NOT_APPLICABLE

Quality gate:
{"PASS" if QUALITY_GATE_PASSED else "FAIL"}

Decision:
{decision}

Next:
66_postE50_multiseries_geometry_pairing.ipynb

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)


EXPERIMENT STATUS

Experiment:
Post-E50 Baseline Inventory

Training performed:
NO

Dataset modification:
NO

Checkpoint modification:
NO

Frozen E50 source modified:
NO

Git branch:
research/post-e50-ai-vnext

Git commit:
90f4a8e076cc18efa9ebce10c604e1c3d1f3e35c

Notebook:
65_postE50_baseline_inventory_and_task_matrix.ipynb

Leakage check:
NOT_APPLICABLE

Quality gate:
PASS

Decision:
BASELINE_FROZEN

Next:
66_postE50_multiseries_geometry_pairing.ipynb

Warnings:
docs/P10_7_RUNTIME_INTEGRATION.md documents REAL_RUNTIME_PREPROCESSING_PARITY_VALIDATED=false, but the current runtime code (disc_degenerative_product_runtime.py) has PREPROCESSING_PARITY_VALIDATED=True. Documentation is stale relative to code; code is treated as source of truth here.

